## Proyecto: Iberdrola DTR

Autor: GTEA. Universidad de Cantabria   
Última revisión: 27/5/2023


**Paquetes generales**

In [1]:
import psycopg2 # Database manipulation
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, date 
from dateutil.relativedelta import relativedelta
from tqdm.notebook import trange, tqdm # Progress bar
from time import sleep
import warnings
import xlrd # Excel manipulation
import configparser # Config file
import matplotlib.pyplot as plt
import plotly.express as px
from dash import dcc
import calendar # operation with dates

# conda activate the environment_you_intend_to_use
# pip install --upgrade --quiet jupyter_client ipywidgets

In [2]:
import sys
print( sys.version)

3.9.15 (main, Nov 24 2022, 14:39:17) [MSC v.1916 64 bit (AMD64)]


**Paquetes específicos DLR**

In [3]:
from cable import cable
from case import case
from ieee738 import ieee738
from cigre601 import cigre601
from pvsystems import pvsystems
import matplotlib.pyplot as plt 


# Needed only during the development phase.
from importlib import reload
reload(  cable)
reload(  case)
reload( ieee738)
reload( cigre601)
reload( pvsystems)

<module 'pvsystems.pvsystems' from 'e:\\mario\\python\\pypacity\\pvsystems\\pvsystems.py'>

**DATOS**

Conductor: LA-280

In [4]:

NSELECT = 2 
CablePE = cable.Cable()
c_db, error = CablePE.load_cable_db()
CablePE.EMISS = 0.8
CablePE.ABSORP = 0.8

PV1 = pvsystems.PVSystems()
Case1 = case.Case()
Case1.demo( NSELECT)
# Ambient conditions
Case1.TAMB = 40.0
Case1.CDR_LAT_DEG = 30
Case1.ALBEDO = 0.1
Case1.beta = 0
Case1.CDR_ELEV = 0
Case1.TCDRPRELOAD = 100
#Case1.TCDRMAX = 150
#Case1.TCDR = 100.0
Case1.WINDANG_DEG = 60
Case1.Z1_DEG = 90
Case1.Ns = 1.0
Case1.SUN_TIME = 11
Case1.NDAY = PV1.DayOfYear( 10, 6) # 10th June
print("NDAY: " + str(Case1.NDAY))

NDAY: 161


Read config parameters from config file "config.ini"

In [5]:
ConfigFile = r".\\config.ini"
configP = configparser.ConfigParser()
configP.read(ConfigFile)

['.\\\\config.ini']

In [6]:
PWD = configP['GENERAL']['PWD']
USER = configP['GENERAL']['USER']
DATABASE = configP['GENERAL']['DATABASE']


In [7]:
def GetNumberValuesMonth( iYear, iMonth, deltaT=5):
    firstday, lastday = calendar.monthrange( iYear, iMonth)
    if np.isnan( deltaT) == 0:
        Nvalues = int(lastday*24*60/deltaT)
    else:
        Nvalues = 0
    return Nvalues


def GetDatesFromYearMonth( iYear, iMonth):
    firstday, lastday = calendar.monthrange( iYear, iMonth)
    firstdate =datetime( iYear, iMonth, 1, hour=0, minute=0, second=0, microsecond=0, tzinfo=None)
    lastdate = datetime( iYear, iMonth, lastday, hour=23, minute=59, second=0, microsecond=0, tzinfo=None)
    #print( firstdate)
    #print( lastdate)
    return firstdate, lastdate


def FilterByDates( DFdata, firstdate, lastdate):
    DFdatafiltered = DFdata.loc[(DFdata['measurementTime'] >= firstdate) & (DFdata['measurementTime'] <= lastdate)]
    return DFdatafiltered


def FilterByNodeID( DFdata, NID):
    DFdatafiltered = DFdata.loc[ DFdata['nodeId'] == NID]
    return DFdatafiltered


def GenerateDatesBetween( firstdate, lastdate):
    fechas = []
    fechasdt = []
    fecha_actual = firstdate

    while fecha_actual <= lastdate:
        fecha_actual_str = fecha_actual.strftime("%Y-%m")
        fechas.append( fecha_actual_str)
        fechasdt.append( datetime.strptime( fecha_actual_str, "%Y-%m"))
        fecha_actual += relativedelta(months=1)

    return fechasdt


def AnalyzeDataByMonth( DFdata, firstdate, lastdate, NodeID):
    dateslist = GenerateDatesBetween( firstdate, lastdate)
    NValT = 0
    NRealValT = 0
    for data in dateslist:
        iyear = data.year
        imonth = data.month
        fd, ld = GetDatesFromYearMonth( iyear, imonth)
        DFf = FilterByDates( rDF, fd, ld)
        DFf = FilterByNodeID( DFf, NodeID)
        deltaT = GetDeltaT( DFf, fd, ld, NodeID)
        NVal = GetNumberValuesMonth( iyear, imonth, deltaT/60)
        NRealVal = len(DFf)
        NValT += NVal
        NRealValT += NRealVal
        print('year: ', iyear, ' ; month: ', imonth, ' ; valores teóricos: ', NVal, ' ; valores medidos: ', NRealVal)
    print("Node: " + str(NodeID) + " / Valores Teóricos: " + str(NValT) + " / Valores medidos: " + str(NRealValT))


def GetDeltaT( DFdata, firstdate, lastdate, NodeID):
    ElPalmarNodeID = [ '10160',  '10166', '10174']
    HellinNodeID = [ '10021', '10031', '10042', '10055', '10068', '10085']
    if NodeID in ElPalmarNodeID:
        return( 60)
    if NodeID in HellinNodeID:
        return( 300)
    
    DeltaTs = []
    dateslist = GenerateDatesBetween( firstdate, lastdate)
    for data in dateslist:
        iyear = data.year
        imonth = data.month
        NVal = GetNumberValuesMonth( iyear, imonth)
        fd, ld = GetDatesFromYearMonth( iyear, imonth)
        DFf = FilterByDates( rDF, fd, ld)
        DFf = FilterByNodeID( DFf, NodeID)
        DFf = DFf[ DFf['ambientTemperature'] != 'NaN']
        if len(DFf)>0:
            d1 = DFf['measurementTime']
            t1 = d1.iloc[0]
            t2 = d1.iloc[1]
            dt = t2-t1
            dt = dt.total_seconds()
            DeltaTs.append( dt)
    if len(DeltaTs)>0:
        AVGDeltaTs = np.average( DeltaTs)
    else:
        AVGDeltaTs = 0
    return( AVGDeltaTs)
                  

In [8]:
DATABASE

'Iberdrola'

Connection to the table geDTR in the database Iberdrola

In [9]:
# Conectar con la BBDD
database = psycopg2.connect( database = DATABASE, user = USER, password = PWD)
database.autocommit = True
cursor = database.cursor()

In [14]:
# Plantilla SQL para leer datos de la BBDD Iberdrola
sqlstr = '''SELECT * FROM public.gedtr
WHERE gedtr."nodeid" = '10160'
AND gedtr."ambienttemperature" IS DISTINCT FROM 'NaN' 
AND gedtr."windspeed" IS DISTINCT FROM 'NaN'
AND gedtr."solarradiation" IS DISTINCT FROM 'NaN'
order by gedtr."measurementtime" ASC;'''

# WHERE linename = 'Hellin-Calasparra' 
# -- where measurementtime > '2021-12-01 00:06:00' and measurementtime < '2021-12-01 00:10:00'

In [15]:
cursor.execute( sqlstr)
results = cursor.fetchall()

In [16]:
type( results)

list

In [17]:
rDF = pd.DataFrame(results)
columnsDF = ['geDTRId',
             'siteName',
             'lineName',
             'nodeId',
             'measurementTime',
             'ambientTemperature',
             'windSpeed',
             'windSpeedavg',
             'windDirection',
             'windDirectionD',
             'solarRadiation',
             'dewPoint',
             'windSonicWindSpeed',
             'windSonicWindDirectionavg',
             'windSonicWindDirectionD',
             'conductorTemperature',
             'conductorCurrent',
             'quality']
rDF.columns = columnsDF

In [18]:
rDF

,geDTRId,siteName,lineName,nodeId,measurementTime,ambientTemperature,windSpeed,windSpeedavg,windDirection,windDirectionD,solarRadiation,dewPoint,windSonicWindSpeed,windSonicWindDirectionavg,windSonicWindDirectionD,conductorTemperature,conductorCurrent,quality
0,1,ElPalmarEspinardo,ElPalmar-Espinardo,10160,2021-12-01 00:00:00,7.1,0.0,0.0,87.0,281.0,0.0,5.5,NaN,NaN,NaN,8.1,120.76,1
1,2,ElPalmarEspinardo,ElPalmar-Espinardo,10160,2021-12-01 00:01:00,7.1,0.0,0.0,83.0,281.0,0.0,5.5,NaN,NaN,NaN,8.0,119.82,1
2,3,ElPalmarEspinardo,ElPalmar-Espinardo,10160,2021-12-01 00:02:00,7.1,0.0,0.0,78.0,258.0,0.0,5.5,NaN,NaN,NaN,8.0,122.19,1
3,4,ElPalmarEspinardo,ElPalmar-Espinardo,10160,2021-12-01 00:03:00,7.1,0.0,0.0,80.0,258.0,0.0,5.5,NaN,NaN,NaN,8.0,120.56,1
4,5,ElPalmarEspinardo,ElPalmar-Espinardo,10160,2021-12-01 00:04:00,7.1,0.0,0.0,85.0,281.0,0.0,5.5,NaN,NaN,NaN,8.0,122.70,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
919543,3319144,ElPalmarEspinardo,ElPalmar-Espinardo,10160,2023-08-31 23:56:00,24.5,0.0,0.0,19.0,213.0,0.0,18.8,NaN,NaN,NaN,24.0,140.47,1
919544,3319145,ElPalmarEspinardo,ElPalmar-Espinardo,10160,2023-08-31 23:57:00,24.5,0.0,0.0,19.0,213.0,0.0,18.8,NaN,NaN,NaN,24.0,135.89,1
919545,3319146,ElPalmarEspinardo,ElPalmar-Espinardo,10160,2023-08-31 23:58:00,24.5,0.0,0.0,19.0,213.0,0.0,18.8,NaN,NaN,NaN,24.0,136.02,1
919546,3319147,ElPalmarEspinardo,ElPalmar-Espinardo,10160,2023-08-31 23:59:00,24.4,0.0,0.0,19.0,213.0,0.0,18.8,NaN,NaN,NaN,24.0,134.28,1


Close the connection

In [19]:
cursor.close()

Number of elements

In [20]:
print(type(  rDF['measurementTime']))
start = datetime(  year=2021, month=12, day=1, hour=1, minute=0, second=0, microsecond=0, tzinfo=None)
end = datetime( year=2022, month=12, day=31, hour=23, minute=55, second=0, microsecond=0, tzinfo=None)
deltat =timedelta( minutes=5)

currenttime = start
timelist = []

while currenttime < end:
    timelist.append( currenttime)
    currenttime += deltat

<class 'pandas.core.series.Series'>


In [21]:
print( len(timelist))


114035


In [22]:
imprimirfigura = 0

if imprimirfigura == 1:
    fig = px.scatter( rDF, x='measurementTime', y='ambientTemperature', color='nodeId')
    fig.update_traces( mode='markers+lines')
    fig.show()

### Obtener el rango temporal de datos en la BBDD

In [23]:
listafechas = rDF['measurementTime']
Primerdato = min(listafechas)
Ultimodato = max(listafechas)
print('Primer dato: ', Primerdato)
print('Último dato: ', Ultimodato)

Primer dato:  2021-12-01 00:00:00
Último dato:  2023-09-01 00:00:00


In [24]:
type( rDF['measurementTime'])

pandas.core.series.Series

In [ ]:
ElPalmarNodeID = [ '10160',  '10166', '10174']
HellinNodeID = [ '10021', '10031', '10042', '10055', '10068', '10085']

In [ ]:
type(rDF['nodeId'])

In [ ]:
for nodes in ElPalmarNodeID:
    d1 = GetDeltaT( rDF, Primerdato, Ultimodato, nodes)
    print(d1)

In [ ]:
for nodes in HellinNodeID:
    d1 = GetDeltaT( rDF, Primerdato, Ultimodato, nodes)
    print(d1)

In [ ]:
print('Línea El Palmar - Espinardo')
for nodes in ElPalmarNodeID:
    print('NodeID: ', nodes)
    AnalyzeDataByMonth( rDF, Primerdato, Ultimodato, nodes)
print('*****************************************')

print('Línea Hellín - Calasparra')
for nodes in HellinNodeID:
    print('NodeID: ', nodes)
    AnalyzeDataByMonth( rDF, Primerdato, Ultimodato, nodes)
print('*****************************************')

In [ ]:
# import the tabulate library
from tabulate import tabulate
# create a list of lists
table = [["Sun",696000,1989100000],["Earth",6371,5973.6], ["Moon",1737,73.5],["Mars",3390,641.85]]
# print the table
print(tabulate(table))
# print the table with headers
print(tabulate(table, headers=["Planet","R (km)","mass (x 10^29 kg)"]))
# print the table with headers and alignment
print(tabulate(table, headers=["Planet","R (km)", "mass (x 10^29 kg)"], tablefmt="grid"))
# print

In [ ]:
from tabulate import tabulate
from texttable import Texttable

import latextable

rows = [['Rocket', 'Organisation', 'LEO Payload (Tonnes)', 'Maiden Flight'],
        ['Saturn V', 'NASA', '140', '1967'],
        ['Space Shuttle', 'NASA', '24.4', '1981'],
        ['Falcon 9 FT-Expended', 'SpaceX', '22.8', '2017'],
        ['Ariane 5 ECA', 'ESA', '21', '2002']]

table = Texttable()
table.set_cols_align(["c"] * 4)
table.set_deco(Texttable.HEADER | Texttable.VLINES)
table.add_rows(rows)

print('Tabulate Table:')
print(tabulate(rows, headers='firstrow'))

print('\nTexttable Table:')
print(table.draw())

print('\nTabulate Latex:')
print(tabulate(rows, headers='firstrow', tablefmt='latex'))

print('\nTexttable Latex:')
print(latextable.draw_latex(table, caption="A comparison of rocket features."))

dcc.Slider(value=4, min=-10, max=20, step=0.5)

from jupyter_plotly_dash import JupyterDash

import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output

app = JupyterDash('SimpleExample')

app.layout = html.Div([
    dcc.RadioItems(
        id='dropdown-color',
        options=[{'label': c, 'value': c.lower()}
                 for c in ['Red', 'Green', 'Blue']],
        value='red'
    ),
    html.Div(id='output-color'),
    dcc.RadioItems(
        id='dropdown-size',
        options=[{'label': i, 'value': j}
                 for i, j in [('L','large'), ('M','medium'), ('S','small')]],
        value='medium'
    ),
    html.Div(id='output-size')

])

@app.callback(
    dash.dependencies.Output('output-color', 'children'),
    [dash.dependencies.Input('dropdown-color', 'value')])
def callback_color(dropdown_value):
    return "The selected color is %s." % dropdown_value

@app.callback(
    dash.dependencies.Output('output-size', 'children'),
    [dash.dependencies.Input('dropdown-color', 'value'),
     dash.dependencies.Input('dropdown-size', 'value')])
def callback_size(dropdown_color, dropdown_size):
    return "The chosen T-shirt is a %s %s one." %(dropdown_size,
                                                  dropdown_color)

app

# Run this app with `python app.py` and
# visit http://127.0.0.1:8050/ in your web browser.

from dash import Dash, html, dcc
import plotly.express as px
import pandas as pd





app = Dash(__name__)

# assume you have a "long-form" data frame
# see https://plotly.com/python/px-arguments/ for more options
df = pd.DataFrame({
    "Fruit": ["Apples", "Oranges", "Bananas", "Apples", "Oranges", "Bananas"],
    "Amount": [4, 1, 2, 2, 4, 5],
    "City": ["SF", "SF", "SF", "Montreal", "Montreal", "Montreal"]
})

fig = px.scatter( rDF, x='measurementTime', y='ambientTemperature', color="nodeId")
fig.update_traces( mode='markers+lines')
#fig.show()

app.layout = html.Div(children=[
    html.H1(children='Iberdrola DTR Project'),

    html.Div(children='''
        Universidad de Cantabria. 
    '''),

    dcc.Graph(
        id='example-graph',
        figure=fig
    )
])

if __name__ == '__main__':
    app.run_server(debug=False)